# 1.追踪```sch.sample_perfect_tile``` 以及 ```auto sch```这一个基本的实现思想
# 2.看看自动程序是否能优化类似flash attention 到 attention2这样的
# 3.meta_sch 做的更复杂的优化 看看sch.trace里面都有什么更复杂的拆分优化 一个搜索空间
# 4.试着对我之前随便写的Conv试一下

In [1]:
import numpy as np
import tvm
from tvm import relax
from tvm.ir.module import IRModule
from tvm.script import relax as R
from tvm.script import tir as T
from rich.syntax import Syntax
from rich import print

syntax = Syntax('code_str', "python", theme="monokai", line_numbers=True)
print(syntax)

  1 code_str                                                                                                       

In [2]:
N, CI, H, W, CO, K = 1, 1, 1026, 1026, 2, 3
OUT_H, OUT_W = H - K + 1, W - K + 1
data = np.arange(N*CI*H*W).reshape(N, CI, H, W)
weight = np.arange(CO*CI*K*K).reshape(CO, CI, K, K)
# data, weight

In [3]:
@tvm.script.ir_module
class MyConv2d:
    @T.prim_func
    def Conv2d( A:T.Buffer((N, CI, H, W), "int64"),
                B:T.Buffer((CO, CI, K, K), "int64"),
                C:T.Buffer((N, CO, OUT_H, OUT_W), "int64"),
                ):
        T.func_attr({"global_symbol":"Conv2d", "tir.noalias":True})
        for b, q, k, i , j, di, dj in T.grid(N, CI, CO, OUT_H, OUT_W, K, K):
            with T.block("Conv"):
                vb, vq, vk, vi, vj, vdi, vdj = T.axis.remap("SSSSSRR", [b, q, k, i, j, di, dj])
                with T.init():
                    C[vb, vk, vi, vj] = T.int64(0)
                C[vb, vk, vi, vj] = A[vb, vq, vi+vdi, vj+vdj] * B[vk, vq, vdi, vdj] + C[vb, vk, vi, vj]



In [4]:
rt_lib = tvm.build(MyConv2d, target="llvm")
data_tvm = tvm.nd.array(data)
weight_tvm = tvm.nd.array(weight)
conv_tvm = tvm.nd.array(np.empty((N, CO, OUT_H, OUT_W), dtype=np.int64))
# rt_lib["Conv2d"](data_tvm, weight_tvm, conv_tvm)
f_timer_before = rt_lib.time_evaluator("Conv2d", tvm.cpu())
print("Time cost of MyModule: %.3f ms" % (f_timer_before(data_tvm, weight_tvm, conv_tvm).mean * 1000))

Time cost of MyModule: 5.566 ms

In [5]:
sch = tvm.tir.Schedule(MyConv2d)
print(sch.mod.script())
block_Conv = sch.get_block("Conv", "Conv2d")
print(type(block_Conv))
print(sch.trace)

# from tvm.script import ir as I
# from tvm.script import tir as T

@I.ir_module
class Module:
    @T.prim_func
    def Conv2d(A: T.Buffer((1, 1, 1026, 1026), "int64"), B: T.Buffer((2, 1, 3, 3), "int64"), C: T.Buffer((1, 2, 
1024, 1024), "int64")):
        T.func_attr({"tir.noalias": T.bool(True)})
        # with T.block("root"):
        for b, q, k, i, j, di, dj in T.grid(1, 1, 2, 1024, 1024, 3, 3):
            with T.block("Conv"):
                vb, vq, vk, vi, vj, vdi, vdj = T.axis.remap("SSSSSRR", )
                T.reads(A, B)
                T.writes(C)
                with T.init():
                    C = T.int64(0)
                C = A * B + C

<class 'tvm.tir.schedule.schedule.BlockRV'>

# from tvm import tir
def apply_trace(sch: tir.Schedule) -> None:
  b0 = sch.get_block(name="Conv", func_name="Conv2d")

# sample_perfect_tile
我这里想法是先优化i j这两个循环维度 因为我想对于conv来说卷积核一般会远远小于输入数据的维度

这里发生的是，每次我们运行 stochastic_schedule_mm 时，它都会随机采样一组不同的 j_factors。我们可以打印出最新的历史轨迹，以查看我们在采样中做出的决定。

i_factors 中的元素并不是实整数。相反，它们是指被采样的随机变量的符号变量。我们可以将这些变量传递给变换的 API 从而指定诸如因子值之类的选择。

In [6]:
def stochastic_schedule_Conv2d(sch: tvm.tir.Schedule):
    block_Conv = sch.get_block("Conv", "Conv2d")
    b, q, k, i , j, di, dj = sch.get_loops(block=block_Conv)

    i_factors = sch.sample_perfect_tile(loop=i, n=2)
    i_0, i_1 = sch.split(loop=i, factors=i_factors)

    j_factors = sch.sample_perfect_tile(loop=j, n=2)
    j_0, j_1 = sch.split(loop=j, factors=j_factors)
    print("type(i_factors):", type(i_factors), "\ttype(i_factors[0])", type(i_factors[0]))
    sch.reorder(b, q, k, i_0 , j_0, i_1, j_1, di, dj)
    sch.decompose_reduction(block_Conv, di)
    return sch

In [7]:
sch = tvm.tir.Schedule(MyConv2d)
sch = stochastic_schedule_Conv2d(sch)
print(sch.mod.script())
print(sch.trace)

type(i_factors): <class 'list'>         type(i_factors[0]) <class 'tvm.tir.expr.Var'>

# from tvm.script import ir as I
# from tvm.script import tir as T

@I.ir_module
class Module:
    @T.prim_func
    def Conv2d(A: T.Buffer((1, 1, 1026, 1026), "int64"), B: T.Buffer((2, 1, 3, 3), "int64"), C: T.Buffer((1, 2, 
1024, 1024), "int64")):
        T.func_attr({"tir.noalias": T.bool(True)})
        # with T.block("root"):
        for b, q, k, i_0, j_0, i_1, j_1 in T.grid(1, 1, 2, 128, 128, 8, 8):
            with T.block("Conv_init"):
                vb, vq, vk = T.axis.remap("SSS", )
                vi = T.axis.spatial(1024, i_0 * 8 + i_1)
                vj = T.axis.spatial(1024, j_0 * 8 + j_1)
                T.reads()
                T.writes(C)
                C = T.int64(0)
            for di, dj in T.grid(3, 3):
                with T.block("Conv_update"):
                    vb, vq, vk = T.axis.remap("SSS", )
                    vi = T.axis.spatial(1024, i_0 * 8 + i_1)
                    vj = T.axis.spatial(1024, j_0 * 8 + j_1)
                    vdi, vdj = T.axis.remap("RR", )
                    T.reads(C, A, B)
                    T.writes(C)
                    C = A * B + C

# from tvm import tir
def apply_trace(sch: tir.Schedule) -> None:
  b0 = sch.get_block(name="Conv", func_name="Conv2d")
  l1, l2, l3, l4, l5, l6, l7 = sch.get_loops(block=b0)
  v8, v9 = sch.sample_perfect_tile(loop=l4, n=2, max_innermost_factor=16, decision=[128, 8])
  l10, l11 = sch.split(loop=l4, factors=[v8, v9], preserve_unit_iters=True, disable_predication=False)
  v12, v13 = sch.sample_perfect_tile(loop=l5, n=2, max_innermost_factor=16, decision=[128, 8])
  l14, l15 = sch.split(loop=l5, factors=[v12, v13], preserve_unit_iters=True, disable_predication=False)
  sch.reorder(l1, l2, l3, l10, l14, l11, l15, l6, l7)
  b16 = sch.decompose_reduction(block=b0, loop=l6)

我们需要一个搜索算法来做到这一点。为了展示这里可以做什么，让我们首先在下面的代码块中尝试最直接的搜索算法——随机搜索。它尝试重复运行 stochastic_schedule_Conv，获取转换后的模块，运行测试，然后保留历史上最好（用时最短）的模块。

In [8]:
def random_search(mod: tvm.IRModule, num_trials=5):
    best_result = None
    best_sch = None

    for i in range(num_trials):
        sch = stochastic_schedule_Conv2d(tvm.tir.Schedule(mod))
        lib = tvm.build(sch.mod, target="llvm")
        f_timer_after = lib.time_evaluator("Conv2d", tvm.cpu())
        result = f_timer_after(data_tvm, weight_tvm, conv_tvm).mean

        print("=====Attempt %d, time-cost: %.3f ms====" % (i, result * 1000))
        print(sch.trace)

        # book keep the best result so far
        if best_result is None or result < best_result:
            best_result = result
            best_sch = sch

    return best_sch

sch = random_search(MyConv2d)
print(sch.trace)

type(i_factors): <class 'list'>         type(i_factors[0]) <class 'tvm.tir.expr.Var'>

=====Attempt 0, time-cost: 5.704 ms====

# from tvm import tir
def apply_trace(sch: tir.Schedule) -> None:
  b0 = sch.get_block(name="Conv", func_name="Conv2d")
  l1, l2, l3, l4, l5, l6, l7 = sch.get_loops(block=b0)
  v8, v9 = sch.sample_perfect_tile(loop=l4, n=2, max_innermost_factor=16, decision=[1024, 1])
  l10, l11 = sch.split(loop=l4, factors=[v8, v9], preserve_unit_iters=True, disable_predication=False)
  v12, v13 = sch.sample_perfect_tile(loop=l5, n=2, max_innermost_factor=16, decision=[512, 2])
  l14, l15 = sch.split(loop=l5, factors=[v12, v13], preserve_unit_iters=True, disable_predication=False)
  sch.reorder(l1, l2, l3, l10, l14, l11, l15, l6, l7)
  b16 = sch.decompose_reduction(block=b0, loop=l6)

type(i_factors): <class 'list'>         type(i_factors[0]) <class 'tvm.tir.expr.Var'>

=====Attempt 1, time-cost: 5.430 ms====

...

# from tvm import tir
def apply_trace(sch: tir.Schedule) -> None:
  b0 = sch.get_block(name="Conv", func_name="Conv2d")
  l1, l2, l3, l4, l5, l6, l7 = sch.get_loops(block=b0)
  v8, v9 = sch.sample_perfect_tile(loop=l4, n=2, max_innermost_factor=16, decision=[512, 2])
  l10, l11 = sch.split(loop=l4, factors=[v8, v9], preserve_unit_iters=True, disable_predication=False)
  v12, v13 = sch.sample_perfect_tile(loop=l5, n=2, max_innermost_factor=16, decision=[512, 2])
  l14, l15 = sch.split(loop=l5, factors=[v12, v13], preserve_unit_iters=True, disable_predication=False)
  sch.reorder(l1, l2, l3, l10, l14, l11, l15, l6, l7)
  b16 = sch.decompose_reduction(block=b0, loop=l6)

type(i_factors): <class 'list'>         type(i_factors[0]) <class 'tvm.tir.expr.Var'>

=====Attempt 4, time-cost: 6.625 ms====

# from tvm import tir
def apply_trace(sch: tir.Schedule) -> None:
  b0 = sch.get_block(name="Conv", func_name="Conv2d")
  l1, l2, l3, l4, l5, l6, l7 = sch.get_loops(block=b0)
  v8, v9 = sch.sample_perfect_tile(loop=l4, n=2, max_innermost_factor=16, decision=[64, 16])
  l10, l11 = sch.split(loop=l4, factors=[v8, v9], preserve_unit_iters=True, disable_predication=False)
  v12, v13 = sch.sample_perfect_tile(loop=l5, n=2, max_innermost_factor=16, decision=[1024, 1])
  l14, l15 = sch.split(loop=l5, factors=[v12, v13], preserve_unit_iters=True, disable_predication=False)
  sch.reorder(l1, l2, l3, l10, l14, l11, l15, l6, l7)
  b16 = sch.decompose_reduction(block=b0, loop=l6)

# from tvm import tir
def apply_trace(sch: tir.Schedule) -> None:
  b0 = sch.get_block(name="Conv", func_name="Conv2d")
  l1, l2, l3, l4, l5, l6, l7 = sch.get_loops(block=b0)
  v8, v9 = sch.sample_perfect_tile(loop=l4, n=2, max_innermost_factor=16, decision=[1024, 1])
  l10, l11 = sch.split(loop=l4, factors=[v8, v9], preserve_unit_iters=True, disable_predication=False)
  v12, v13 = sch.sample_perfect_tile(loop=l5, n=2, max_innermost_factor=16, decision=[1024, 1])
  l14, l15 = sch.split(loop=l5, factors=[v12, v13], preserve_unit_iters=True, disable_predication=False)
  sch.reorder(l1, l2, l3, l10, l14, l11, l15, l6, l7)
  b16 = sch.decompose_reduction(block=b0, loop=l6)

meta_schedule 是支持搜索可能变换空间的命名空间。Meta-Schedule 在幕后做了很多额外的事情：

跨越多个进程的并行基准测试。

使用代价模型 (cost model) 来避免每次都进行基准测试。

基于历史轨迹进行遗传搜索 (evolutionary search)，而不是每次都随机采样。

尽管有这些工具，但我们关键思想是保持不变的：使用随机变换来指定好的程序的搜索空间，使用 ``tune_tir`` API 帮助在搜索空间内搜索并找到最优的调度变换。

这里搜索的时候需要定义元张量算子函数名字为main (可能默认的candidate是只有main这一个列表只能这样)

In [9]:
from tvm import meta_schedule as ms
@tvm.script.ir_module
class MyConv2d:
    @T.prim_func
    def main( A:T.Buffer((N, CI, H, W), "int64"),
                B:T.Buffer((CO, CI, K, K), "int64"),
                C:T.Buffer((N, CO, OUT_H, OUT_W), "int64"),
                ):
        T.func_attr({"global_symbol":"Conv2d", "tir.noalias":True})
        for b, q, k, i , j, di, dj in T.grid(N, CI, CO, OUT_H, OUT_W, K, K):
            with T.block("Conv"):
                vb, vq, vk, vi, vj, vdi, vdj = T.axis.remap("SSSSSRR", [b, q, k, i, j, di, dj])
                with T.init():
                    C[vb, vk, vi, vj] = T.int64(0)
                C[vb, vk, vi, vj] = A[vb, vq, vi+vdi, vj+vdj] * B[vk, vq, vdi, vdj] + C[vb, vk, vi, vj]

def stochastic_schedule_Conv2d(sch: tvm.tir.Schedule):
    block_Conv = sch.get_block("Conv", "main")
    b, q, k, i , j, di, dj = sch.get_loops(block=block_Conv)

    i_factors = sch.sample_perfect_tile(loop=i, n=2)
    i_0, i_1 = sch.split(loop=i, factors=i_factors)

    j_factors = sch.sample_perfect_tile(loop=j, n=2)
    j_0, j_1 = sch.split(loop=j, factors=j_factors)
    sch.reorder(b, q, k, i_0 , j_0, i_1, j_1, di, dj)
    sch.decompose_reduction(block_Conv, di)
    return sch

database = ms.tune_tir(
    mod=MyConv2d,
    target="llvm --num-cores=4",
    max_trials_global=64,
    num_trials_per_iter=64,
    space=ms.space_generator.ScheduleFn(stochastic_schedule_Conv2d), #指定一个搜索空间
    work_dir="./tune_tmp"
)

sch = ms.tir_integration.compile_tir(database, MyConv2d, "llvm --num-cores=4")

2025-03-13 18:00:30 [INFO] Logging directory: ./tune_tmp/logs
2025-03-13 18:00:44 [INFO] LocalBuilder: max_workers = 6
2025-03-13 18:00:45 [INFO] LocalRunner: max_workers = 1
2025-03-13 18:00:46 [INFO] [task_scheduler.cc:159] Initializing Task #0: "main"


,Name,FLOP,Weight,Speed (GFLOPS),Latency (us),Weighted Latency (us),Trials,Done
0,main,37748736,1,N/A,N/A,N/A,0,


2025-03-13 18:00:46 [DEBUG] [task_scheduler.cc:318] 
 ID | Name |     FLOP | Weight | Speed (GFLOPS) | Latency (us) | Weighted Latency (us) | Trials | Done 
-------------------------------------------------------------------------------------------------------
  0 | main | 37748736 |      1 |            N/A |          N/A |                   N/A |      0 |      
-------------------------------------------------------------------------------------------------------
Total trials: 0
Total latency (us): 0


Total trials: 0
Total latency (us): 0

2025-03-13 18:00:46 [INFO] [task_scheduler.cc:180] TaskScheduler picks Task #0: "main"
2025-03-13 18:00:47 [INFO] [task_scheduler.cc:193] Sending 25 sample(s) to builder
2025-03-13 18:00:49 [INFO] [task_scheduler.cc:195] Sending 25 sample(s) to runner
2025-03-13 18:00:58 [DEBUG] XGB iter   0: tr-p-rmse: 0.364429	tr-a-peak@32: 0.995719	tr-rmse: 0.367210	tr-rmse: 0.367210
2025-03-13 18:00:58 [DEBUG] XGB iter  25: tr-p-rmse: 0.043553	tr-a-peak@32: 0.9

,Name,FLOP,Weight,Speed (GFLOPS),Latency (us),Weighted Latency (us),Trials,Done
0,main,37748736,1,7.1554,5275.5924,5275.5924,25,



Total trials: 25
Total latency (us): 5275.59

2025-03-13 18:00:58 [DEBUG] [task_scheduler.cc:318] 
 ID | Name |     FLOP | Weight | Speed (GFLOPS) | Latency (us) | Weighted Latency (us) | Trials | Done 
-------------------------------------------------------------------------------------------------------
  0 | main | 37748736 |      1 |         7.1554 |    5275.5924 |             5275.5924 |     25 |      
-------------------------------------------------------------------------------------------------------
Total trials: 25
Total latency (us): 5275.59

2025-03-13 18:00:58 [INFO] [task_scheduler.cc:180] TaskScheduler picks Task #0: "main"
2025-03-13 18:00:59 [INFO] [task_scheduler.cc:193] Sending 0 sample(s) to builder
2025-03-13 18:00:59 [INFO] [task_scheduler.cc:195] Sending 0 sample(s) to runner
2025-03-13 18:00:59 [INFO] [task_scheduler.cc:237] [Updated] Task #0: "main"


...

2025-03-13 18:01:01 [DEBUG] [task_scheduler.cc:318] 
 ID | Name |     FLOP | Weight | Speed (GFLOPS) | Latency (us) | Weighted Latency (us) | Trials | Done 
-------------------------------------------------------------------------------------------------------
  0 | main | 37748736 |      1 |         7.1554 |    5275.5924 |             5275.5924 |     25 |      
-------------------------------------------------------------------------------------------------------
Total trials: 25
Total latency (us): 5275.59


Total trials: 25
Total latency (us): 5275.59

2025-03-13 18:01:01 [INFO] [task_scheduler.cc:180] TaskScheduler picks Task #0: "main"
2025-03-13 18:01:02 [INFO] [task_scheduler.cc:193] Sending 0 sample(s) to builder
2025-03-13 18:01:02 [INFO] [task_scheduler.cc:195] Sending 0 sample(s) to runner
2025-03-13 18:01:02 [INFO] [task_scheduler.cc:237] [Updated] Task #0: "main"


,Name,FLOP,Weight,Speed (GFLOPS),Latency (us),Weighted Latency (us),Trials,Done
0,main,37748736,1,7.1554,5275.5924,5275.5924,25,


2025-03-13 18:01:02 [DEBUG] [task_scheduler.cc:318] 
 ID | Name |     FLOP | Weight | Speed (GFLOPS) | Latency (us) | Weighted Latency (us) | Trials | Done 
-------------------------------------------------------------------------------------------------------
  0 | main | 37748736 |      1 |         7.1554 |    5275.5924 |             5275.5924 |     25 |      
-------------------------------------------------------------------------------------------------------
Total trials: 25
Total latency (us): 5275.59


Total trials: 25
Total latency (us): 5275.59

2025-03-13 18:01:02 [INFO] [task_scheduler.cc:180] TaskScheduler picks Task #0: "main"
2025-03-13 18:01:03 [INFO] [task_scheduler.cc:260] Task #0 has finished. Remaining task(s): 0


,Name,FLOP,Weight,Speed (GFLOPS),Latency (us),Weighted Latency (us),Trials,Done
0,main,37748736,1,7.1554,5275.5924,5275.5924,25,Y



Total trials: 25
Total latency (us): 5275.59

2025-03-13 18:01:03 [DEBUG] [task_scheduler.cc:318] 
 ID | Name |     FLOP | Weight | Speed (GFLOPS) | Latency (us) | Weighted Latency (us) | Trials | Done 
-------------------------------------------------------------------------------------------------------
  0 | main | 37748736 |      1 |         7.1554 |    5275.5924 |             5275.5924 |     25 |    Y 
-------------------------------------------------------------------------------------------------------
Total trials: 25
Total latency (us): 5275.59



同时可以看到这里的优化并没有带来多少性能的提升可能10%都不足

这里有个隐藏小Bug:f_timer_after = lib.time_evaluator("main", tvm.cpu()) 查看lib中的entry_name可以得知是 "__tvm_main__"


In [10]:
print(sch.trace)

lib = tvm.build(sch.mod, target="llvm")
print(lib.entry_name)
f_timer_after = lib.time_evaluator("__tvm_main__", tvm.cpu())
time_before = f_timer_before(data_tvm, weight_tvm, conv_tvm).mean
time_after = f_timer_after(data_tvm, weight_tvm, conv_tvm).mean
print("Time cost of MyModule before tuning: %.3f ms" % (time_before * 1000))
print("Time cost of MyModule after tuning: %.3f ms" % (time_after * 1000))
print("Time tuning rate is %.3f %% " % ((time_before - time_after)/time_before*100))

# from tvm import tir
def apply_trace(sch: tir.Schedule) -> None:
  b0 = sch.get_block(name="Conv", func_name="main")
  l1, l2, l3, l4, l5, l6, l7 = sch.get_loops(block=b0)
  v8, v9 = sch.sample_perfect_tile(loop=l4, n=2, max_innermost_factor=16, decision=[64, 16])
  l10, l11 = sch.split(loop=l4, factors=[v8, v9], preserve_unit_iters=True, disable_predication=False)
  v12, v13 = sch.sample_perfect_tile(loop=l5, n=2, max_innermost_factor=16, decision=[64, 16])
  l14, l15 = sch.split(loop=l5, factors=[v12, v13], preserve_unit_iters=True, disable_predication=False)
  sch.reorder(l1, l2, l3, l10, l14, l11, l15, l6, l7)
  b16 = sch.decompose_reduction(block=b0, loop=l6)
  sch.enter_postproc()

__tvm_main__

Time cost of MyModule before tuning: 5.703 ms

Time cost of MyModule after tuning: 5.345 ms

Time tuning rate is 6.275 %

# 利用默认的自动调度

在上一节中，我们展示了如何使用我们精心设计的随机变换来优化 IRModule 的计算。Meta-Schedule 带有内置通用随机变换集合，能够适用于广泛的 TensorIR 计算。

这种方法也称为自动调度 (auto-scheduling)，因为搜索空间是由系统生成的。我们可以通过删除行 space=ms.space_generator.ScheduleFn(stochastic_schedule_Conv) 来运行它。

在底层，Meta-Schedule 分析每个 TensorIR block 的数据访问和循环模式，并提出对程序的随机变换方式。

我们不会在本章中讨论这些通用的变换，但要注意它们也只是随机转换加上代码分析而已。我们可以使用上一节中学到的相同机制来增强自动调度。我们将在以后的章节中触及这个主题。

In [11]:
database = ms.tune_tir(
    mod=MyConv2d,
    target="llvm --num-cores=4",
    max_trials_global=64,
    num_trials_per_iter=64,
    work_dir="./tune_tmp"
)
sch = ms.tir_integration.compile_tir(database, MyConv2d, "llvm --num-cores=1")

2025-03-13 18:01:03 [INFO] Logging directory: ./tune_tmp/logs
2025-03-13 18:01:03 [INFO] LocalBuilder: max_workers = 6
2025-03-13 18:01:04 [INFO] LocalRunner: max_workers = 1
2025-03-13 18:01:05 [INFO] [task_scheduler.cc:159] Initializing Task #0: "main"


,Name,FLOP,Weight,Speed (GFLOPS),Latency (us),Weighted Latency (us),Trials,Done
0,main,37748736,1,N/A,N/A,N/A,0,



Total trials: 0
Total latency (us): 0

2025-03-13 18:01:05 [DEBUG] [task_scheduler.cc:318] 
 ID | Name |     FLOP | Weight | Speed (GFLOPS) | Latency (us) | Weighted Latency (us) | Trials | Done 
-------------------------------------------------------------------------------------------------------
  0 | main | 37748736 |      1 |            N/A |          N/A |                   N/A |      0 |      
-------------------------------------------------------------------------------------------------------
Total trials: 0
Total latency (us): 0

2025-03-13 18:01:05 [INFO] [task_scheduler.cc:180] TaskScheduler picks Task #0: "main"
2025-03-13 18:01:09 [INFO] [task_scheduler.cc:193] Sending 64 sample(s) to builder
2025-03-13 18:01:22 [INFO] [task_scheduler.cc:195] Sending 64 sample(s) to runner
2025-03-13 18:01:45 [DEBUG] XGB iter   0: tr-p-rmse: 0.384613	tr-a-peak@32: 1.000000	tr-rmse: 0.195584	tr-rmse: 0.195584
2025-03-13 18:01:45 [DEBUG] XGB iter  25: tr-p-rmse: 0.056539	tr-a-peak@32: 1.0

,Name,FLOP,Weight,Speed (GFLOPS),Latency (us),Weighted Latency (us),Trials,Done
0,main,37748736,1,22.5815,1671.6644,1671.6644,64,



Total trials: 64
Total latency (us): 1671.66

2025-03-13 18:01:45 [DEBUG] [task_scheduler.cc:318] 
 ID | Name |     FLOP | Weight | Speed (GFLOPS) | Latency (us) | Weighted Latency (us) | Trials | Done 
-------------------------------------------------------------------------------------------------------
  0 | main | 37748736 |      1 |        22.5815 |    1671.6644 |             1671.6644 |     64 |      
-------------------------------------------------------------------------------------------------------
Total trials: 64
Total latency (us): 1671.66

2025-03-13 18:01:45 [INFO] [task_scheduler.cc:260] Task #0 has finished. Remaining task(s): 0


,Name,FLOP,Weight,Speed (GFLOPS),Latency (us),Weighted Latency (us),Trials,Done
0,main,37748736,1,22.5815,1671.6644,1671.6644,64,Y



Total trials: 64
Total latency (us): 1671.66

2025-03-13 18:01:45 [DEBUG] [task_scheduler.cc:318] 
 ID | Name |     FLOP | Weight | Speed (GFLOPS) | Latency (us) | Weighted Latency (us) | Trials | Done 
-------------------------------------------------------------------------------------------------------
  0 | main | 37748736 |      1 |        22.5815 |    1671.6644 |             1671.6644 |     64 |    Y 
-------------------------------------------------------------------------------------------------------
Total trials: 64
Total latency (us): 1671.66



In [12]:
lib = tvm.build(sch.mod, target="llvm")
f_timer_after = lib.time_evaluator("__tvm_main__", tvm.cpu())
print("Time cost of MyModule after tuning: %.3f ms" % (f_timer_after(data_tvm, weight_tvm, conv_tvm).mean * 1000))
print(sch.trace)
print(sch.mod.script())

Time cost of MyModule after tuning: 2.088 ms

# from tvm import tir
def apply_trace(sch: tir.Schedule) -> None:
  b0 = sch.get_block(name="Conv", func_name="main")
  b1 = sch.get_block(name="root", func_name="main")
  sch.annotate(block_or_loop=b0, ann_key="meta_schedule.tiling_structure", ann_val="SSRSRS")
  l2, l3, l4, l5, l6, l7, l8 = sch.get_loops(block=b0)
  v9, v10, v11, v12 = sch.sample_perfect_tile(loop=l2, n=4, max_innermost_factor=64, decision=[1, 1, 1, 1])
  l13, l14, l15, l16 = sch.split(loop=l2, factors=[v9, v10, v11, v12], preserve_unit_iters=True, 
disable_predication=False)
  v17, v18, v19, v20 = sch.sample_perfect_tile(loop=l3, n=4, max_innermost_factor=64, decision=[1, 1, 1, 1])
  l21, l22, l23, l24 = sch.split(loop=l3, factors=[v17, v18, v19, v20], preserve_unit_iters=True, 
disable_predication=False)
  v25, v26, v27, v28 = sch.sample_perfect_tile(loop=l4, n=4, max_innermost_factor=64, decision=[1, 2, 1, 1])
  l29, l30, l31, l32 = sch.split(loop=l4, factors=[v25, v26, v27, v28], preserve_unit_iters=True, 
disable_predication=False)
  v33, v34, v35, v36 = sch.sample_perfect_tile(loop=l5, n=4, max_innermost_factor=64, decision=[256, 1, 2, 2])
  l37, l38, l39, l40 = sch.split(loop=l5, factors=[v33, v34, v35, v36], preserve_unit_iters=True, 
disable_predication=False)
  v41, v42, v43, v44 = sch.sample_perfect_tile(loop=l6, n=4, max_innermost_factor=64, decision=[32, 32, 1, 1])
  l45, l46, l47, l48 = sch.split(loop=l6, factors=[v41, v42, v43, v44], preserve_unit_iters=True, 
disable_predication=False)
  v49, v50 = sch.sample_perfect_tile(loop=l7, n=2, max_innermost_factor=64, decision=[3, 1])
  l51, l52 = sch.split(loop=l7, factors=[v49, v50], preserve_unit_iters=True, disable_predication=False)
  v53, v54 = sch.sample_perfect_tile(loop=l8, n=2, max_innermost_factor=64, decision=[1, 3])
  l55, l56 = sch.split(loop=l8, factors=[v53, v54], preserve_unit_iters=True, disable_predication=False)
  sch.reorder(l13, l21, l29, l37, l45, l14, l22, l30, l38, l46, l51, l55, l15, l23, l31, l39, l47, l52, l56, l16, 
l24, l32, l40, l48)
  b57 = sch.cache_write(block=b0, write_buffer_index=0, storage_scope="global")
  sch.reverse_compute_at(block=b57, loop=l46, preserve_unit_loops=True, index=-1)
  sch.annotate(block_or_loop=b1, ann_key="meta_schedule.parallel", ann_val=64)
  sch.annotate(block_or_loop=b1, ann_key="meta_schedule.vectorize", ann_val=64)
  v58 = sch.sample_categorical(candidates=[0, 16, 64, 512], probs=[0.25, 0.25, 0.25, 0.25], decision=1)
  sch.annotate(block_or_loop=b1, ann_key="meta_schedule.unroll_explicit", ann_val=v58)
  sch.enter_postproc()
  b59 = sch.get_block(name="root", func_name="main")
  sch.unannotate(block_or_loop=b59, ann_key="meta_schedule.parallel")
  sch.unannotate(block_or_loop=b59, ann_key="meta_schedule.vectorize")
  sch.unannotate(block_or_loop=b59, ann_key="meta_schedule.unroll_explicit")
  b60, b61 = sch.get_child_blocks(b59)
  l62, l63, l64, l65, l66, l67, l68, l69, l70, l71, l72, l73, l74, l75, l76, l77, l78, l79, l80, l81, l82, l83, 
l84, l85 = sch.get_loops(block=b60)
  l86 = sch.fuse(l62, l63, l64, l65, preserve_unit_iters=True)
  sch.parallel(loop=l86)
  sch.annotate(block_or_loop=l86, ann_key="pragma_auto_unroll_max_step", ann_val=16)
  sch.annotate(block_or_loop=l86, ann_key="pragma_unroll_explicit", ann_val=1)
  l87, l88, l89, l90, l91, l92, l93, l94, l95, l96, l97 = sch.get_loops(block=b61)
  b98 = sch.get_block(name="Conv", func_name="main")
  l99, l100, l101, l102, l103, l104, l105, l106, l107, l108, l109, l110, l111, l112, l113, l114, l115, l116, l117, 
l118, l119 = sch.get_loops(block=b98)
  b120 = sch.decompose_reduction(block=b98, loop=l106)

# from tvm.script import ir as I
# from tvm.script import tir as T

@I.ir_module
class Module:
    @T.prim_func
    def main(A: T.Buffer((1, 1, 1026, 1026), "int64"), B: T.Buffer((2, 1, 3, 3), "int64"), C: T.Buffer((1, 2, 1024,
1024), "int64")):
        T.func_attr({"global_symbol": "Conv2d", "tir.noalias": T.bool(True)})
        # with T.block("root"):
        C_global = T.alloc_buffer((1, 2, 1024, 1024), "int64")
        for b_0_q_0_k_0_i_0_fused in T.parallel(256, annotations={"pragma_auto_unroll_max_step": 16, 
"pragma_unroll_explicit": 1}):
            for j_0, b_1, q_1, k_1, i_1, j_1 in T.grid(32, 1, 1, 2, 1, 32):
                for b_2_init, q_2_init, k_2_init, i_2_init, j_2_init, b_3_init, q_3_init, k_3_init, i_3_init, 
j_3_init in T.grid(1, 1, 1, 2, 1, 1, 1, 1, 2, 1):
                    with T.block("Conv_init"):
                        vb = T.axis.spatial(1, b_1 + b_2_init + b_3_init)
                        vq = T.axis.spatial(1, q_1 + q_2_init + q_3_init)
                        vk = T.axis.spatial(2, k_1 + k_2_init + k_3_init)
                        vi = T.axis.spatial(1024, b_0_q_0_k_0_i_0_fused * 4 + i_1 * 4 + i_2_init * 2 + i_3_init)
                        vj = T.axis.spatial(1024, j_0 * 32 + j_1 + j_2_init + j_3_init)
                        T.reads()
                        T.writes(C_global)
                        T.block_attr({"meta_schedule.tiling_structure": "SSRSRS"})
                        C_global = T.int64(0)
                for di_0, dj_0, b_2, q_2, k_2, i_2, j_2, di_1, dj_1, b_3, q_3, k_3, i_3, j_3 in T.grid(3, 1, 1, 1, 
1, 2, 1, 1, 3, 1, 1, 1, 2, 1):
                    with T.block("Conv_update"):
                        vb = T.axis.spatial(1, b_1 + b_2 + b_3)
                        vq = T.axis.spatial(1, q_1 + q_2 + q_3)
                        vk = T.axis.spatial(2, k_1 + k_2 + k_3)
                        vi = T.axis.spatial(1024, b_0_q_0_k_0_i_0_fused * 4 + i_1 * 4 + i_2 * 2 + i_3)
                        vj = T.axis.spatial(1024, j_0 * 32 + j_1 + j_2 + j_3)
                        vdi = T.axis.reduce(3, di_0 + di_1)
                        vdj = T.axis.reduce(3, dj_0 * 3 + dj_1)
                        T.reads(C_global, A, B)
                        T.writes(C_global)
                        T.block_attr({"meta_schedule.tiling_structure": "SSRSRS"})
                        C_global = A * B + C_global
                for ax0, ax1, ax2, ax3 in T.grid(1, 1, 4, 1):
                    with T.block("C_global"):
                        v0 = T.axis.spatial(1, ax0)
                        v1 = T.axis.spatial(2, k_1 + ax1)
                        v2 = T.axis.spatial(1024, b_0_q_0_k_0_i_0_fused * 4 + ax2)
                        v3 = T.axis.spatial(1024, j_0 * 32 + j_1 + ax3)
                        T.reads(C_global)
                        T.writes(C)
                        C = C_global